---
## Section 0 - Repository Structure & Execution

> **Read this before running any cells.** This notebook is the final modelling
> layer of a two-stage pipeline. Preprocessing scripts were
> executed first to eventually generate `cleaned_master.csv`.

### 0.1 Full Directory Tree

```
vct-team-prediction/
├── aggregated/                        ← Generated by src/ scripts
│   ├── cleaned_master.csv             ← PRIMARY INPUT for this notebook
│   ├── master_dataset.csv
│   ├── scores.csv
│   ├── overview.csv
│   ├── eco_stats.csv
│   ├── kills_stats.csv
│   ├── draft_phase.csv
│   ├── maps_scores.csv
│   ├── maps_played.csv
│   ├── teams_picked_agents.csv
│   ├── players_stats.csv
│   └── tournaments_stages_matches_games_ids.csv
├── data/                              ← Raw match logs
│   ├── vct_2023/matches/*.csv
│   ├── vct_2024/matches/*.csv
│   └── vct_2025/matches/*.csv
├── src/                               ← Preprocessing scripts
│   ├── data_agg.py                    ← Step 1: merges raw tournament CSVs
│   ├── master_dataset_creation.py     ← Step 2: joins all aggregated files
│   └── clean_master_file.py           ← Step 3: cleans & adds target variable
├── vct-team-prediction.ipynb          ← Step 4: ML modelling (This notebook)
├── section(1,2,3,4,5,6,7,8,0)
├── requirements.txt
└── README.md
```

### 0.2 Pipeline Execution Order

This notebook **requires** `cleaned_master.csv` to exist before
running. That file is produced by three preprocessing scripts in `src/`. It is automatically downloaded from our Github when running this notebook.


### 0.3 How the Pipeline Feeds This Notebook

| Script | Input | Output | What it does |
|--------|-------|--------|--------------|
| `data_agg.py` | `data/vct_*/matches/*.csv` | `aggregated/*.csv` | Concatenates the 2023, 2024, 2025 tournament files for each stat category into a single file per category. Adds a `tournament` column for season labelling. |
| `master_dataset_creation.py` | `aggregated/*.csv` | `aggregated/master_dataset.csv` | Merges all stat-category files on `(Match Name, Tournament, Stage, Match Type, Team)`. Pivots each team into `ta_*` / `tb_*` prefixed columns. |
| `clean_master_file.py` | `aggregated/master_dataset.csv` | `aggregated/cleaned_master.csv` | Drops rows with missing `ta_avg_acs` / `tb_avg_acs` / KD values; imputes remaining nulls with column medians; sets eco/draft/kill nulls to 0; **creates the binary target** `team_a_won` from the `Match Result` string. |
| `vct-team-prediction.ipynb` | `aggregated/cleaned_master.csv` | Model results & plots | **This file.** Loads `cleaned_master.csv`, engineers 13 pre-match features, trains Logistic Regression / Random Forest / XGBoost, and evaluates on the 2025 holdout season. |

### 0.4 Schema of `cleaned_master.csv` (47 columns)

The file written by `clean_master_file.py` has the following column groups. All
columns used as direct model inputs are marked **bold**.

| Group | Columns | dtype |
|-------|---------|-------|
| Match metadata | `Tournament`, `Stage`, `Match Type`, `Match Name`, `Match ID` | str / int |
| Team identifiers | `Team A`, `Team B`, `Team A Score`, `Team B Score`, `Match Result` | str / int |
| Team A performance | `ta_avg_rating`, `ta_avg_acs`, `ta_avg_kd`, `ta_avg_adr`, `ta_avg_fk`, `ta_avg_hs` | float |
| Team A economy | `ta_eco_$ (won)`, `ta_eco_$$ (won)`, `ta_eco_$$$ (won)`, `ta_eco_Eco (won)`, `ta_eco_Pistol Won` | float |
| Team A multi-kills | `ta_total_2k` … `ta_total_5k`, `ta_total_clutches` | float |
| Team A draft | **`ta_draft_ban`**, `ta_draft_pick` | float |
| Team B (mirror) | `tb_avg_rating` … `tb_draft_pick` | float |
| **Target** | **`team_a_won`** | int (0/1) |

> **Note on `ta_draft_ban`:** This notebook uses `ta_draft_ban == 1` to derive
> the binary feature `ta_ban_first` (Section 2.6). The value `1` indicates Team A
> held the first ban slot in the draft phase - sourced directly from
> `draft_phase.csv` via `master_dataset_creation.py`.


---
